# TFG · Lección 3 (circuitos cuánticos): comprobaciones en Qiskit
Cada bloque comprueba un resultado del Capítulo 3 del cuaderno. Ejecuta las celdas **en orden**.

In [ ]:
from qiskit import __version__
print(__version__)

In [ ]:
import numpy as np
from numpy import sqrt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

sampler = StatevectorSampler()

def contar(circuito, shots=1000):
    """Ejecuta un circuito con medidas (measure_all) y devuelve el recuento de resultados."""
    resultado = sampler.run([circuito], shots=shots).result()[0]
    return resultado.data.meas.get_counts()

## Experimento 1 · Un qubit: el circuito $H\to S\to H\to T$
Debe dar $THSH|0\rangle = \frac{1+i}{2}|0\rangle + \frac{1}{\sqrt2}|1\rangle$.

In [ ]:
qc = QuantumCircuit(1)
qc.h(0); qc.s(0); qc.h(0); qc.t(0)
display(qc.draw(output="mpl"))

estado = Statevector.from_label("0").evolve(qc)
print(estado.draw("latex_source"))
esperado = Statevector(np.array([(1 + 1j) / 2, 1 / sqrt(2)]))
print("Coincide con el cálculo a mano:", estado.equiv(esperado))

# El circuito se lee al revés que la fórmula: su matriz es T H S H
H, S, T = Operator.from_label("H"), Operator.from_label("S"), Operator.from_label("T")
print("Matriz del circuito == T·H·S·H:", Operator.from_circuit(qc).equiv(Operator(T.data @ H.data @ S.data @ H.data)))

## Experimento 2 · El circuito de Bell y la convención de orden
Hadamard sobre el qubit de **arriba** ($q_0$, que es $\mathsf{Y}$) y CNOT con control $q_0$ y objetivo $q_1$ ($\mathsf{X}$).
La matriz debe ser $U = CX\,(\mathbb{I}\otimes H)$ y cumplir
$U|00\rangle = |\phi^+\rangle$, $U|01\rangle = |\phi^-\rangle$, $U|10\rangle = |\psi^+\rangle$, $U|11\rangle = -|\psi^-\rangle$.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
display(bell.draw(output="mpl"))
bell.draw(output="mpl").savefig("circuito_bell_cap3.png", dpi=200, bbox_inches="tight")

U = Operator.from_circuit(bell).data
print("sqrt(2)·U =")
print(np.round(sqrt(2) * U, 3).real)

I2 = np.eye(2); Hm = np.array([[1, 1], [1, -1]]) / sqrt(2)
CX = np.array([[1,0,0,0],[0,0,0,1],[0,0,1,0],[0,1,0,0]])
print("U == CX (I ⊗ H):", np.allclose(U, CX @ np.kron(I2, Hm)))

bell_estados = {"00": [1,0,0,1], "01": [1,0,0,-1], "10": [0,1,1,0], "11": [0,1,-1,0]}
for etiqueta, v in bell_estados.items():
    salida = Statevector.from_label(etiqueta).evolve(bell)
    print(f"U|{etiqueta}> = {salida.draw('latex_source')}")

## Experimento 3 · Medir el estado de Bell
Al medir $|\phi^+\rangle$ solo deben salir `00` y `11`, cada uno aproximadamente la mitad de las veces.

In [ ]:
bell_medido = bell.copy()
bell_medido.measure_all()
display(bell_medido.draw(output="mpl"))

cuentas = contar(bell_medido, shots=1000)
print(cuentas)
fig = plot_histogram(cuentas)
fig.savefig("histograma_bell.png", dpi=200, bbox_inches="tight")
fig

## Experimento 4 · Productos internos, fidelidad y proyecciones

In [ ]:
psi = Statevector([(1 + 2j) / 3, -2 / 3])
mas = Statevector.from_label("+")

# Producto interno: np.vdot conjuga el PRIMER vector
print("<psi|+> · 3√2 =", np.round(np.vdot(psi.data, mas.data) * 3 * sqrt(2), 6))   # -1-2i
print("<+|psi> · 3√2 =", np.round(np.vdot(mas.data, psi.data) * 3 * sqrt(2), 6))   # -1+2i (conjugado)

# Fidelidad |<psi|phi>|^2 (entre 0 y 1)
from qiskit.quantum_info import state_fidelity
print("Fidelidad |<0|+>|^2 =", round(state_fidelity(Statevector.from_label("0"), mas), 4))   # 0.5
print("Fidelidad |<+|->|^2 =", round(state_fidelity(mas, Statevector.from_label("-")), 4))  # 0

# La proyección sobre |+> es hermítica e idempotente
P = np.outer(mas.data, mas.data.conj())
print("Pi_+ =\n", np.round(P, 3).real)
print("Hermítica:", np.allclose(P, P.conj().T), "   Idempotente:", np.allclose(P @ P, P))

## Experimento 5 · Medir en la base $\{|+\rangle, |-\rangle\}$ = aplicar $H$ y medir
Con $|0\rangle$ deben salir 0 y 1 al 50 %; con $|+\rangle$, siempre 0 (resultado "+").

In [ ]:
for nombre, preparar in [("|0>", []), ("|+>", ["h"])]:
    qc = QuantumCircuit(1)
    for g in preparar:
        getattr(qc, g)(0)
    qc.h(0)            # cambio a la base {|+>, |->}
    qc.measure_all()
    print(f"Estado {nombre}: {contar(qc)}")

## Experimento 6 · Fase global frente a fase relativa

In [ ]:
menos = Statevector.from_label("-")
print("|-> y -|-> iguales salvo fase global:", menos.equiv(Statevector(-menos.data)))
print("|+> y |-> iguales salvo fase global:  ", mas.equiv(menos))
print("|<+|->| =", abs(np.vdot(mas.data, menos.data)))   # 0: son ortogonales

## Experimento 7 · No clonación: el CNOT copia $|0\rangle$ y $|1\rangle$, pero no $|+\rangle$
Circuito: el qubit a copiar es $q_1$ (abajo) y el qubit "en blanco" es $q_0$ (arriba), inicializado en $|0\rangle$.

In [ ]:
clonador = QuantumCircuit(2)
clonador.cx(1, 0)

for nombre in ["0", "1", "+"]:
    entrada = Statevector.from_label(nombre + "0")          # |psi> ⊗ |0>
    salida = entrada.evolve(clonador)
    deseado = Statevector.from_label(nombre + nombre)       # |psi> ⊗ |psi>
    print(f"|{nombre}>: salida = {salida.draw('latex_source'):35s} ¿es |{nombre}>|{nombre}>? {salida.equiv(deseado)}")

## Experimento 8 · Estimar un kernel cuántico con un circuito
Codificamos un dato $x$ con una rotación: $|\phi(x)\rangle = R_y(x)|0\rangle$.
El circuito $R_y(y)$ seguido de $R_y(x)^\dagger = R_y(-x)$ y una medición da `0` con probabilidad $K(x,y) = |\langle\phi(x)|\phi(y)\rangle|^2 = \cos^2\frac{x-y}{2}$.

In [ ]:
def kernel_exacto(x, y):
    fx = Statevector.from_label("0").evolve(QuantumCircuit(1).compose(_ry(x)))
    fy = Statevector.from_label("0").evolve(QuantumCircuit(1).compose(_ry(y)))
    return abs(np.vdot(fx.data, fy.data)) ** 2

def _ry(t):
    qc = QuantumCircuit(1); qc.ry(t, 0); return qc

def kernel_estimado(x, y, shots):
    qc = QuantumCircuit(1)
    qc.ry(y, 0)        # U(y)
    qc.ry(-x, 0)       # U(x)^dagger
    qc.measure_all()
    cuentas = contar(qc, shots)
    return cuentas.get("0", 0) / shots

x, y = 0.3, 1.2
print("Exacto (fórmula):   ", round(np.cos((x - y) / 2) ** 2, 4))
print("Exacto (productos): ", round(kernel_exacto(x, y), 4))
for shots in [100, 1000, 10000]:
    print(f"Estimado con {shots:5d} disparos: {kernel_estimado(x, y, shots):.4f}")